In [1]:
import MDAnalysis as mda
import numpy as np
from tqdm import tqdm

/Users/nate_schwindt/anaconda3/envs/mda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
membrane_bounds = { # as determined by the intersection of the polymer and water density profiles
    'Pristine': np.array([ 26.50190379,  49.39451567,  72.28712755,  95.17973943,   118.07235131]),
    'NaOCl': np.array([ 24.32690572,  47.28358504,  70.24026436,  93.19694368,   116.153623  ]),
    'NaOCl + NaOH': np.array([ 5.21002256, 28.0388828 , 50.86774304, 73.69660328, 96.52546351])
}

In [3]:
def average_element_counts(counts_per_frame):
    totals = {}
    n_frames = len(counts_per_frame)

    for elements, counts in counts_per_frame:
        for element, count in zip(elements, counts):
            totals[element] = totals.get(element, 0) + int(count)

    return {element: total / n_frames for element, total in sorted(totals.items())}

## Pristine

In [4]:
label = 'Pristine'
path = './pristine'
u = mda.Universe(f'{path}/prod.tpr', f'{path}/prod_centered.xtc')

C = u.select_atoms('resname PA* and element C')
O = u.select_atoms('resname PA* and element O')
N = u.select_atoms('resname PA* and element N')
Cl = u.select_atoms('resname PA* and element Cl')

cooh_c = u.select_atoms('(type c) and (not bonded type n)')

polymer_counts = []
other_counts = []
for ts in tqdm(u.trajectory):
    inside_atoms = u.select_atoms(f'prop z > {membrane_bounds[label][0]} and prop z < {membrane_bounds[label][-1]}')
    polymer_inside = inside_atoms.select_atoms('resname PA* and not element H')
    other_inside = inside_atoms.select_atoms('not resname PA* and not element H')
    
    polymer_counts.append(np.unique(polymer_inside.elements, return_counts=True))
    other_counts.append(np.unique(other_inside.elements, return_counts=True))

100%|██████████| 4001/4001 [00:48<00:00, 82.12it/s]


In [5]:
n_polymer = len(u.select_atoms('resname PA* and not element H'))
for at in [C, O, N, Cl]:
    if len(at) > 0:
        print(f'{at[0].element} in PA: {len(at)} atoms, {len(at)/n_polymer*100:.1f}% percent')

print('\nO/N ratio:', len(O)/len(N))
print(f'COOH in PA: {len(cooh_c)} atoms, {len(cooh_c)/n_polymer*100:.1f}% percent')
print(f'DC, N/(N+COOH)*100%: {len(N)/(len(N)+len(cooh_c))*100:.1f}%\n')

polymer_avg_counts = average_element_counts(polymer_counts)
other_avg_counts = average_element_counts(other_counts)
total_atoms = sum(polymer_avg_counts.values()) + sum(other_avg_counts.values())

print('\nAverage composition inside membrane bounds:')
for element, count in polymer_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in polymer: {count:.1f} atoms, {percent:.1f}% of total')

for element, count in other_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in other: {count:.1f} atoms, {percent:.1f}% of total')

total_O = polymer_avg_counts['O'] + other_avg_counts['O']
print(f'\nTotal O atoms inside membrane bounds: {total_O:.1f}, {total_O/total_atoms*100:.1f}% of total')

C in PA: 8688 atoms, 74.2% percent
O in PA: 1594 atoms, 13.6% percent
N in PA: 1432 atoms, 12.2% percent

O/N ratio: 1.1131284916201116
COOH in PA: 130 atoms, 1.1% percent
DC, N/(N+COOH)*100%: 91.7%


Average composition inside membrane bounds:
C in polymer: 8086.3 atoms, 58.8% of total
N in polymer: 1337.3 atoms, 9.7% of total
O in polymer: 1475.6 atoms, 10.7% of total
Cl in other: 3.1 atoms, 0.0% of total
Na in other: 53.3 atoms, 0.4% of total
O in other: 2788.5 atoms, 20.3% of total

Total O atoms inside membrane bounds: 4264.2, 31.0% of total


## Chlorinated

In [6]:
label = 'NaOCl'
path = './chlorinated'
u = mda.Universe(f'{path}/prod.tpr', f'{path}/prod_centered.xtc')

C = u.select_atoms('resname PA* and element C')
O = u.select_atoms('resname PA* and element O')
N = u.select_atoms('resname PA* and element N')
Cl = u.select_atoms('resname PA* and element Cl')

cooh_c = u.select_atoms('(type c) and (not bonded type n)')

polymer_counts = []
other_counts = []
for ts in tqdm(u.trajectory):
    inside_atoms = u.select_atoms(f'prop z > {membrane_bounds[label][0]} and prop z < {membrane_bounds[label][-1]}')
    polymer_inside = inside_atoms.select_atoms('resname PA* and not element H')
    other_inside = inside_atoms.select_atoms('not resname PA* and not element H')
    
    polymer_counts.append(np.unique(polymer_inside.elements, return_counts=True))
    other_counts.append(np.unique(other_inside.elements, return_counts=True))

100%|██████████| 2001/2001 [00:27<00:00, 73.40it/s]


In [7]:
n_polymer = len(u.select_atoms('resname PA* and not element H'))
for at in [C, O, N, Cl]:
    print(f'{at[0].element} in PA: {len(at)} atoms, {len(at)/n_polymer*100:.1f}% percent')

print('\nO/N ratio:', len(O)/len(N))
print(f'COOH in PA: {len(cooh_c)} atoms, {len(cooh_c)/n_polymer*100:.1f}% percent')
print(f'DC, N/(N+COOH)*100%: {len(N)/(len(N)+len(cooh_c))*100:.1f}%\n')

polymer_avg_counts = average_element_counts(polymer_counts)
other_avg_counts = average_element_counts(other_counts)
total_atoms = sum(polymer_avg_counts.values()) + sum(other_avg_counts.values())

print('\nAverage composition inside membrane bounds:')
for element, count in polymer_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in polymer: {count:.1f} atoms, {percent:.1f}% of total')

for element, count in other_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in other: {count:.1f} atoms, {percent:.1f}% of total')

total_O = polymer_avg_counts['O'] + other_avg_counts['O']
print(f'\nTotal O atoms inside membrane bounds: {total_O:.1f}, {total_O/total_atoms*100:.1f}% of total')

total_Cl = polymer_avg_counts['Cl'] + other_avg_counts['Cl']
print(f'Total Cl atoms inside membrane bounds: {total_Cl:.1f}, {total_Cl/total_atoms*100:.1f}% of total')

C in PA: 8688 atoms, 70.5% percent
O in PA: 1594 atoms, 12.9% percent
N in PA: 1432 atoms, 11.6% percent
Cl in PA: 618 atoms, 5.0% percent

O/N ratio: 1.1131284916201116
COOH in PA: 130 atoms, 1.1% percent
DC, N/(N+COOH)*100%: 91.7%


Average composition inside membrane bounds:
C in polymer: 8122.8 atoms, 56.4% of total
Cl in polymer: 565.6 atoms, 3.9% of total
N in polymer: 1341.3 atoms, 9.3% of total
O in polymer: 1480.8 atoms, 10.3% of total
Cl in other: 4.2 atoms, 0.0% of total
Na in other: 53.8 atoms, 0.4% of total
O in other: 2830.5 atoms, 19.7% of total

Total O atoms inside membrane bounds: 4311.3, 29.9% of total
Total Cl atoms inside membrane bounds: 569.8, 4.0% of total


## Chlorinated + alkaline

In [8]:
label = 'NaOCl + NaOH'
path = './chlorinated_alkaline'
u = mda.Universe(f'{path}/prod.tpr', f'{path}/prod_centered.xtc')

C = u.select_atoms('resname PA* and element C')
O = u.select_atoms('resname PA* and element O')
N = u.select_atoms('resname PA* and element N')
Cl = u.select_atoms('resname PA* and element Cl')

cooh_c = u.select_atoms('(type c) and (not bonded type n)')

polymer_counts = []
other_counts = []
for ts in tqdm(u.trajectory):
    inside_atoms = u.select_atoms(f'prop z > {membrane_bounds[label][0]} and prop z < {membrane_bounds[label][-1]}')
    polymer_inside = inside_atoms.select_atoms('resname PA* and not element H')
    other_inside = inside_atoms.select_atoms('not resname PA* and not element H')
    
    polymer_counts.append(np.unique(polymer_inside.elements, return_counts=True))
    other_counts.append(np.unique(other_inside.elements, return_counts=True))

100%|██████████| 2001/2001 [00:25<00:00, 77.56it/s]


In [9]:
n_polymer = len(u.select_atoms('resname PA* and not element H'))
for at in [C, O, N, Cl]:
    print(f'{at[0].element} in PA: {len(at)} atoms, {len(at)/n_polymer*100:.1f}% percent')

print('\nO/N ratio:', len(O)/len(N))
print(f'COOH in PA: {len(cooh_c)} atoms, {len(cooh_c)/n_polymer*100:.1f}% percent')
print(f'DC, N/(N+COOH)*100%: {len(N)/(len(N)+len(cooh_c))*100:.1f}%\n')

polymer_avg_counts = average_element_counts(polymer_counts)
other_avg_counts = average_element_counts(other_counts)
total_atoms = sum(polymer_avg_counts.values()) + sum(other_avg_counts.values())

print('\nAverage composition inside membrane bounds:')
for element, count in polymer_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in polymer: {count:.1f} atoms, {percent:.1f}% of total')

for element, count in other_avg_counts.items():
    percent = count / total_atoms * 100
    print(f'{element} in other: {count:.1f} atoms, {percent:.1f}% of total')

total_O = polymer_avg_counts['O'] + other_avg_counts['O']
print(f'\nTotal O atoms inside membrane bounds: {total_O:.1f}, {total_O/total_atoms*100:.1f}% of total')

total_Cl = polymer_avg_counts['Cl'] + other_avg_counts['Cl']
print(f'Total Cl atoms inside membrane bounds: {total_Cl:.1f}, {total_Cl/total_atoms*100:.1f}% of total')

C in PA: 8688 atoms, 72.6% percent
O in PA: 1783 atoms, 14.9% percent
N in PA: 1432 atoms, 12.0% percent
Cl in PA: 60 atoms, 0.5% percent

O/N ratio: 1.2451117318435754
COOH in PA: 319 atoms, 2.7% percent
DC, N/(N+COOH)*100%: 81.8%


Average composition inside membrane bounds:
C in polymer: 8205.4 atoms, 57.4% of total
Cl in polymer: 53.8 atoms, 0.4% of total
N in polymer: 1358.4 atoms, 9.5% of total
O in polymer: 1648.8 atoms, 11.5% of total
Cl in other: 3.5 atoms, 0.0% of total
Na in other: 125.0 atoms, 0.9% of total
O in other: 2902.9 atoms, 20.3% of total

Total O atoms inside membrane bounds: 4551.7, 31.8% of total
Total Cl atoms inside membrane bounds: 57.4, 0.4% of total
